In [0]:
# ============================================================
# GOLD LAYER: Fun facts and aggregations
# ============================================================
# One table per insight. Each table is tiny (KBs) and ready 
# to be pushed to Neon Postgres for the API layer.
# ============================================================

from pyspark.sql.functions import col, count, countDistinct
from pyspark.sql.window import Window

SILVER_WATCH = "youtube_wrapped.silver.watch_history_clean"
SILVER_VIDEOS = "youtube_wrapped.silver.video_enrichment"
SILVER_ARTISTS = "youtube_wrapped.silver.artist_enrichment"
GOLD_SCHEMA = "youtube_wrapped.gold"

df_watch = spark.table(SILVER_WATCH)
df_videos = spark.table(SILVER_VIDEOS)
df_artists = spark.table(SILVER_ARTISTS)

print(f"Watches: {df_watch.count():,}")
print(f"Videos enriched: {df_videos.count():,}")
print(f"Artists enriched: {df_artists.count():,}")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_overview AS
    SELECT
      COUNT(*) AS total_watches,
      COUNT(DISTINCT video_id) AS unique_videos,
      COUNT(DISTINCT channel_id) AS unique_channels,
      SUM(CASE WHEN is_music_topic THEN 1 ELSE 0 END) AS music_watches,
      COUNT(DISTINCT CASE WHEN is_music_topic THEN artist_name_from_topic END) AS unique_artists,
      MIN(watched_at_utc) AS first_watch,
      MAX(watched_at_utc) AS latest_watch,
      DATEDIFF(MAX(watched_at_utc), MIN(watched_at_utc)) AS days_of_history,
      ROUND(100.0 * SUM(CASE WHEN is_music_topic THEN 1 ELSE 0 END) / COUNT(*), 1) AS music_pct
    FROM {SILVER_WATCH}
""")

spark.table(f"{GOLD_SCHEMA}.fact_overview").show(truncate=False)

In [0]:
# Top 20 music artists
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_top_artists AS
    SELECT
      artist_name_from_topic AS artist_name,
      COUNT(*) AS plays,
      COUNT(DISTINCT video_id) AS unique_videos
    FROM {SILVER_WATCH}
    WHERE is_music_topic = true
    GROUP BY artist_name_from_topic
    ORDER BY plays DESC
    LIMIT 20
""")

# Top 20 channels overall
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_top_channels AS
    SELECT
      channel_id,
      channel_name,
      COUNT(*) AS watches,
      COUNT(DISTINCT video_id) AS unique_videos
    FROM {SILVER_WATCH}
    WHERE channel_id IS NOT NULL AND channel_id != ''
    GROUP BY channel_id, channel_name
    ORDER BY watches DESC
    LIMIT 20
""")

print("✅ fact_top_artists:")
spark.table(f"{GOLD_SCHEMA}.fact_top_artists").show(10, truncate=False)
print("✅ fact_top_channels:")
spark.table(f"{GOLD_SCHEMA}.fact_top_channels").show(10, truncate=False)

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_top_genres AS
    WITH exploded AS (
      SELECT a.artist_name, explode(a.tags) AS raw_tag
      FROM {SILVER_ARTISTS} a
      WHERE a.tags IS NOT NULL AND size(a.tags) > 0
    ),
    artist_tags AS (
      SELECT e.artist_name, lower(trim(e.raw_tag)) AS tag, w.video_id
      FROM exploded e
      JOIN {SILVER_WATCH} w ON e.artist_name = w.artist_name_from_topic
      WHERE w.is_music_topic = true
    ),
    genre_tags AS (
      SELECT * FROM artist_tags
      WHERE tag NOT IN (
        'indian','norwegian','norway','pakistani','american','british',
        'canadian','australian','japanese','korean','french','spanish',
        'mexican','german','irish','scottish','swedish',
        'female vocalists','male vocalists','singer-songwriter',
        'theatre','asian','world fusion','world',
        'academy award winner','cham cham','ar-rahman','los angeles'
      )
      AND tag NOT RLIKE '^[0-9]{{2,4}}s?$'
      AND tag RLIKE '^[a-z0-9 \\\\-/&]+$'
    )
    SELECT 
      tag AS genre,
      COUNT(DISTINCT artist_name) AS artists,
      COUNT(*) AS total_plays
    FROM genre_tags
    GROUP BY tag
    ORDER BY total_plays DESC
    LIMIT 20
""")

print("✅ fact_top_genres:")
spark.table(f"{GOLD_SCHEMA}.fact_top_genres").show(20, truncate=False)

In [0]:
DESI_TAGS = "'filmi','bollywood','indian pop','bhangra','punjabi','hindi','desi','desi rap','film composer','tamil','telugu','marathi','sufi','qawwali','ghazal','classical indian','hindustani'"

spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_genre_split AS
    WITH exploded AS (
      SELECT 
        a.artist_name, 
        explode(a.tags) AS raw_tag
      FROM {SILVER_ARTISTS} a
      WHERE a.tags IS NOT NULL AND size(a.tags) > 0
    ),
    artist_tags AS (
      SELECT 
        artist_name,
        lower(trim(raw_tag)) AS tag
      FROM exploded
    ),
    artist_classification AS (
      SELECT 
        artist_name,
        MAX(CASE WHEN tag IN ({DESI_TAGS}) THEN 1 ELSE 0 END) AS is_desi
      FROM artist_tags
      GROUP BY artist_name
    ),
    classified_watches AS (
      SELECT 
        w.video_id,
        CASE 
          WHEN c.is_desi = 1 THEN 'desi'
          WHEN c.artist_name IS NOT NULL THEN 'western'
          ELSE 'untagged'
        END AS music_category
      FROM {SILVER_WATCH} w
      LEFT JOIN artist_classification c
        ON w.artist_name_from_topic = c.artist_name
      WHERE w.is_music_topic = true
    )
    SELECT 
      music_category,
      COUNT(*) AS plays,
      ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM classified_watches
    GROUP BY music_category
    ORDER BY plays DESC
""")

print("✅ fact_genre_split:")
spark.table(f"{GOLD_SCHEMA}.fact_genre_split").show()

In [0]:
# By hour of day
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_listening_by_hour AS
    SELECT watch_hour, COUNT(*) AS watches
    FROM {SILVER_WATCH}
    GROUP BY watch_hour
    ORDER BY watch_hour
""")

# By day of week
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_listening_by_dayofweek AS
    SELECT watch_day_of_week, watch_day_name, COUNT(*) AS watches
    FROM {SILVER_WATCH}
    GROUP BY watch_day_of_week, watch_day_name
    ORDER BY watch_day_of_week
""")

# Monthly timeline
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_listening_timeline AS
    SELECT 
      watch_year,
      watch_month,
      DATE_FORMAT(MIN(watched_at_utc), 'yyyy-MM') AS year_month,
      COUNT(*) AS watches,
      SUM(CASE WHEN is_music_topic THEN 1 ELSE 0 END) AS music_watches
    FROM {SILVER_WATCH}
    GROUP BY watch_year, watch_month
    ORDER BY watch_year, watch_month
""")

print("✅ Time aggregations written")
spark.table(f"{GOLD_SCHEMA}.fact_listening_timeline").show(20)

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_main_character_artist AS
    WITH artist_totals AS (
      SELECT 
        artist_name_from_topic AS artist_name,
        COUNT(*) AS total_plays
      FROM {SILVER_WATCH}
      WHERE is_music_topic = true
      GROUP BY artist_name_from_topic
    ),
    artist_by_month AS (
      SELECT 
        artist_name_from_topic AS artist_name,
        DATE_FORMAT(watched_at_utc, 'yyyy-MM') AS year_month,
        COUNT(*) AS plays
      FROM {SILVER_WATCH}
      WHERE is_music_topic = true
      GROUP BY artist_name_from_topic, DATE_FORMAT(watched_at_utc, 'yyyy-MM')
    ),
    peak_month_per_artist AS (
      SELECT 
        artist_name,
        year_month AS peak_month,
        plays AS peak_month_plays,
        ROW_NUMBER() OVER (PARTITION BY artist_name ORDER BY plays DESC) AS rn
      FROM artist_by_month
    )
    SELECT 
      t.artist_name,
      t.total_plays,
      p.peak_month,
      p.peak_month_plays,
      ROUND(100.0 * p.peak_month_plays / t.total_plays, 1) AS peak_month_pct_of_artist
    FROM artist_totals t
    JOIN peak_month_per_artist p 
      ON t.artist_name = p.artist_name AND p.rn = 1
    ORDER BY t.total_plays DESC
    LIMIT 1
""")

print("✅ fact_main_character_artist:")
spark.table(f"{GOLD_SCHEMA}.fact_main_character_artist").show(truncate=False)

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_binge_sessions AS
    WITH watches_with_gap AS (
      SELECT 
        watched_at_utc,
        video_title,
        channel_name,
        UNIX_TIMESTAMP(watched_at_utc) - UNIX_TIMESTAMP(
          LAG(watched_at_utc) OVER (ORDER BY watched_at_utc)
        ) AS gap_seconds
      FROM {SILVER_WATCH}
    ),
    sessions AS (
      SELECT 
        *,
        SUM(CASE WHEN gap_seconds > 1800 OR gap_seconds IS NULL THEN 1 ELSE 0 END) 
          OVER (ORDER BY watched_at_utc) AS session_id
      FROM watches_with_gap
    )
    SELECT 
      session_id,
      MIN(watched_at_utc) AS session_start,
      MAX(watched_at_utc) AS session_end,
      COUNT(*) AS videos_in_session,
      ROUND((UNIX_TIMESTAMP(MAX(watched_at_utc)) - UNIX_TIMESTAMP(MIN(watched_at_utc))) / 60.0, 1) AS duration_minutes
    FROM sessions
    GROUP BY session_id
    HAVING COUNT(*) >= 3
    ORDER BY videos_in_session DESC
    LIMIT 10
""")

print("✅ fact_binge_sessions:")
spark.table(f"{GOLD_SCHEMA}.fact_binge_sessions").show(truncate=False)

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_night_owl_score AS
    SELECT 
      ROUND(100.0 * SUM(CASE WHEN watch_hour BETWEEN 0 AND 4 THEN 1 ELSE 0 END) / COUNT(*), 1) 
        AS night_owl_pct,
      ROUND(100.0 * SUM(CASE WHEN watch_hour BETWEEN 5 AND 11 THEN 1 ELSE 0 END) / COUNT(*), 1) 
        AS morning_pct,
      ROUND(100.0 * SUM(CASE WHEN watch_hour BETWEEN 12 AND 17 THEN 1 ELSE 0 END) / COUNT(*), 1) 
        AS afternoon_pct,
      ROUND(100.0 * SUM(CASE WHEN watch_hour BETWEEN 18 AND 23 THEN 1 ELSE 0 END) / COUNT(*), 1) 
        AS evening_pct
    FROM {SILVER_WATCH}
""")

print("✅ fact_night_owl_score:")
spark.table(f"{GOLD_SCHEMA}.fact_night_owl_score").show()

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_SCHEMA}.fact_rediscovered_artists AS
    WITH artist_watches AS (
      SELECT 
        artist_name_from_topic AS artist_name,
        watched_at_utc,
        LAG(watched_at_utc) OVER (PARTITION BY artist_name_from_topic ORDER BY watched_at_utc) AS prev_watch
      FROM {SILVER_WATCH}
      WHERE is_music_topic = true
    ),
    gaps AS (
      SELECT 
        artist_name,
        DATEDIFF(watched_at_utc, prev_watch) AS gap_days,
        prev_watch AS last_watched_before_gap,
        watched_at_utc AS rediscovered_at
      FROM artist_watches
      WHERE prev_watch IS NOT NULL AND DATEDIFF(watched_at_utc, prev_watch) >= 180
    )
    SELECT * FROM gaps
    ORDER BY gap_days DESC
    LIMIT 10
""")

print("✅ fact_rediscovered_artists:")
spark.table(f"{GOLD_SCHEMA}.fact_rediscovered_artists").show(truncate=False)

In [0]:
%sql
SHOW TABLES IN youtube_wrapped.gold;